# Setup

In [0]:
#Imports

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.stat import Correlation

from pyspark.sql.types import IntegerType, FloatType, DoubleType, LongType, ShortType, DecimalType
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, expr, col, when, lit, regexp_extract, percentile_approx, regexp_replace, expr, count, to_timestamp, unix_timestamp, mean as _mean, stddev as _stddev

from pyspark.sql.window import Window

import holidays

import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

# Checkpointing Info
section = "3"
number = "1"
folder_path = f"dbfs:/student-groups/Group_{section}_{number}"
dbutils.fs.mkdirs(folder_path)

# OTPW Base

In [0]:
 # OTPW
# df_otpw = spark.read.format("csv").option("header","true").load(f"{data_BASE_DIR}/OTPW_3M_2015.csv")
df_otpw = spark.read.format("csv").option("header","true").load(f"{data_BASE_DIR}/OTPW_12M/OTPW_12M/OTPW_12M_2015.csv.gz")
display(df_otpw.limit(10))

In [0]:
# Creating initial df for the feature engineering pipeline
df_otpw_engr = df_otpw.withColumn("FL_DATE", to_timestamp(col("FL_DATE"), "yyyy-MM-dd")).withColumn("DEP_DEL15", col("DEP_DEL15").cast("double").cast("boolean"))

# Drop rows where DEP_DEL15 is null
df_otpw_engr = df_otpw_engr.filter(col("DEP_DEL15").isNotNull())
df_otpw_engr = df_otpw_engr.cache()

print(df_otpw_engr.count())
display(df_otpw_engr.limit(10))

## Helper Functions

In [0]:
### This run_logistic_regression function will be used as a dummy function to ensure that the columns being selected as part of feature engineering are correctly selected for the model training

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn import metrics

def evaluate(lr_model, split_df):
    preds = lr_model.transform(split_df)
    unweighted_f1_macro = metrics.f1_score(
        y_true=preds.select("label").collect(),
        y_pred=preds.select("prediction").collect(),
        average="macro"
    )
    return unweighted_f1_macro

def run_logistic_regression(df, features=None, label_col="DEP_DEL15", test_ratio=0.2, seed=42):
    # Default features: all columns except label
    if features is None:
        features = [c for c in df.columns if c != label_col and c != "FL_DATE" and c != "MONTH"]

        # Assemble features
        assembler = VectorAssembler(inputCols=features, outputCol="features", handleInvalid="skip")
        df = assembler.transform(df).withColumn("label", col(label_col).cast("double"))

    train_df, test_df = df.randomSplit([1 - test_ratio, test_ratio], seed=seed)

    # Fit logistic regression
    lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
    lr_model = lr.fit(train_df)

    # Predict and evaluate
    print("Train")
    train_unweighted_f1 = evaluate(lr_model, train_df)
    print("F1: ", train_unweighted_f1)
    print("Test")
    test_unweighted_f1 = evaluate(lr_model, test_df)
    print("F1: ", test_unweighted_f1)

    metrics = spark.createDataFrame([
        ("train", train_unweighted_f1),
        ("test", test_unweighted_f1)
    ], ["split", "unweighted f1"])
    display(metrics)
    return lr_model, metrics


In [0]:

def cleanWeatherColumns(df_otpw_engr_weather, weather_cols_numeric, weather_cols_categorical):
    # Clean columns
    # Known non-numeric patterns in NOAA weather data:
    #   '*' : quality flag (trailing e.g. '29.94*', or standalone -> null)
    #   's' : suspect value suffix (e.g. '46s')
    #   'T' : trace precipitation (≈ 0, treated as 0)
    #   'VRB' : variable wind direction (treated as null)

    for c in weather_cols_numeric:
        df_otpw_engr_weather = (
            df_otpw_engr_weather
            # standalone *, VRB -> null; T (trace precip) -> '0'
            .withColumn(c, when(col(c) == '*', lit(None))
                        .when(col(c) == 'T', lit('0'))
                        .when(col(c) == 'VRB', lit(None))
                        .otherwise(col(c)))
            # strip trailing * and s flags 
            .withColumn(c, regexp_replace(col(c), r'[*s]+$', ''))
            # extract numeric part
            .withColumn(c, regexp_extract(col(c), r'([+-]?\d+\.?\d*)', 1))
            # empty -> null
            .withColumn(c, when(col(c) == '', lit(None)).otherwise(col(c)))
            # cast to double
            # .withColumn(c, col(c).cast('double'))
            .withColumn(c, expr(f"try_cast(`{c}` as double)"))
        )


    # Verify null counts 
    total = df_otpw_engr_weather.count()
    non_null_counts = df_otpw_engr_weather.select(
        *[count(when(col(c).isNotNull(), 1)).alias(c) for c in weather_cols_numeric]
    ).toPandas().T.reset_index()
    non_null_counts.columns = ['column', 'non_null']
    non_null_counts['null'] = total - non_null_counts['non_null']
    non_null_counts['null_pct'] = (non_null_counts['null'] / total * 100).round(1)

    print(f"Total rows: {total:,}\n")
    # display(non_null_counts)
    # df_otpw_engr_weather.printSchema()
    return df_otpw_engr_weather



In [0]:
def _class_(df):
    def fit(df):
        self.imputation_number # 8
        return
    def transform(df):
        df = df.withColumn("imputation_number", )
        return df
    return _class_

#save in pickle

#_class.transform(df_val)

In [0]:
def cleanFlightsColumns(df_otpw_engr_flights, flights_cols_numeric, flights_cols_categorical):
    # Clean and cast numeric flight columns
    for c in flights_cols_numeric:
        if c in df_otpw_engr_flights.columns:
            df_otpw_engr_flights = df_otpw_engr_flights.withColumn(c, expr(f"try_cast(`{c}` as double)"))

    # Clean and cast categorical flight columns
    for c in flights_cols_categorical:
        if c in df_otpw_engr_flights.columns:
            df_otpw_engr_flights = df_otpw_engr_flights.withColumn(c, expr(f"try_cast(`{c}` as int)"))

    # Drop rows where any selected flight column is null
    critical_cols = [
        "OP_UNIQUE_CARRIER", "ORIGIN",
        "QUARTER", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "CRS_DEP_TIME", "YEAR"
    ]
    df_otpw_engr_flights = df_otpw_engr_flights.dropna(subset=[c for c in critical_cols if c in df_otpw_engr_flights.columns])

    display(df_otpw_engr_flights)
    df_otpw_engr_flights.printSchema()
    print(df_otpw_engr_flights.count())
    return df_otpw_engr_flights


In [0]:
# Data Imputation

def impute_weather_columns(df):
    """
    Impute nulls in weather columns.
    Three tiers:
      1. Zero-fill   -> columns where null = "nothing noteworthy happened"
      2. Domain fill  -> ceiling_height_ft null = clear sky (99999)
      3. Median fill  -> slow-moving continuous variables with low null rates
    """

    # ---- 1. Zero-fill: null means the phenomenon wasn't observed ----
    zero_fill_cols = {
        "HourlyWindGustSpeed": 0,   # gusts only reported above ~14 kt
        "HourlyPrecipitation": 0,   # blank = dry conditions
        "HourlyPressureChange": 0,  # blank = stable pressure
        "HourlyWindDirection": 0,   # blank = calm / no direction
    }
    for c, fill_val in zero_fill_cols.items():
        if c in df.columns:
            df = df.withColumn(c, when(col(c).isNull(), lit(fill_val)).otherwise(col(c)))

    # ---- 2. Domain default: ceiling_height_ft ----
    # No sky-condition report -> assume unlimited ceiling (clear sky)
    if "ceiling_height_ft" in df.columns:
        df = df.withColumn(
            "ceiling_height_ft",
            when(col("ceiling_height_ft").isNull(), lit(99999))
            .otherwise(col("ceiling_height_ft"))
        )

    # ---- 3. Median fill: continuous measurements with low null rates ----
    # These are slow-moving physical quantities. Median is a safe to use.
    median_fill_cols = [
        "HourlySeaLevelPressure",    # 9.6% null
        "HourlyVisibility",          # 0.25%
        "HourlyWindSpeed",           # 0.29%
        "HourlyDryBulbTemperature",  # 0.26%
        "HourlyDewPointTemperature", # 0.27%
        "HourlyRelativeHumidity",    # 0.28%
        "HourlyPressureTendency"
    ]
    existing = [c for c in median_fill_cols if c in df.columns]

    if existing:
        # Compute approximate medians 
        medians_row = df.select(
            *[percentile_approx(col(c), 0.5).alias(c) for c in existing]
        ).collect()[0]

        for c in existing:
            med_val = medians_row[c]
            if med_val is not None:
                df = df.withColumn(c, when(col(c).isNull(), lit(med_val)).otherwise(col(c)))
                print(f"  {c}: filled nulls with median = {med_val}")

    return df

In [0]:
def fillNullFlightsColumns(df):
    ## Fill nulls before encoding categorical engineered features

    categorical_cols = [
        "ORIGIN",
        "DEST",
        "ORIGIN_STATE_ABR",
        "DEST_STATE_ABR"
    ]

    numeric_fill_map = {
        "scheduled_flights_before_current": 0,
        "cum_delay_rate_today": 0.0,
        "days_to_nearest_holiday": 999
    }

    categorical_fill_map = {c: "missing" for c in categorical_cols}

    df = df.fillna(numeric_fill_map)
    df = df.fillna(categorical_fill_map)
    return df

In [0]:
def featureEngineeredFlights(df):

    ## Order of current flight in today's schedule in airport X

    window_day = Window.partitionBy("FL_DATE").orderBy("CRS_DEP_TIME")

    df = df.withColumn(
        "departure_sequence",
        F.row_number().over(window_day)
    )

    ## Accumulated scheduled flights for today

    window_cum_flights = (
        Window.partitionBy("ORIGIN", "FL_DATE")
            .orderBy("CRS_DEP_TIME", "OP_CARRIER_FL_NUM")
            .rowsBetween(Window.unboundedPreceding, -1)
    )

    df = df.withColumn(
        "scheduled_flights_before_current",
        F.count(F.lit(1)).over(window_cum_flights)
    )

    df = df.fillna(
        {"scheduled_flights_before_current": 0}
    )
    print("Created scheduled_flights_before_current")

    ## Early-morning delay propagation

    # Extract hour from HHMM format
    df = df.withColumn(
        "dep_hour",
        (F.col("CRS_DEP_TIME") / 100).cast("int")
    )

    # Early morning flag
    df = df.withColumn(
        "is_early_morning",
        F.col("dep_hour") < 8
    )

    morning_rates = (
        df.filter(F.col("is_early_morning"))
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("morning_delay_rate_airport_day")
        )
    )

    # Historical average morning delay rate by airport
    airport_avg_morning_delay = (
        morning_rates
        .groupBy("ORIGIN")
        .agg(F.avg("morning_delay_rate_airport_day").alias("avg_morning_delay_rate_airport"))
    )

    # Add airport benchmark to morning rates
    morning_rates = morning_rates.join(
        airport_avg_morning_delay,
        on="ORIGIN",
        how="left"
    )

    # Join back to full dataset
    df = df.join(
        morning_rates.select(
            "ORIGIN",
            "FL_DATE",
            "morning_delay_rate_airport_day",
            "avg_morning_delay_rate_airport"
        ),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    # Final classification
    df = df.withColumn(
        "morning_delay_propagation",
        F.when(F.col("is_early_morning"), F.lit("early-morning flight"))
        .when(F.col("morning_delay_rate_airport_day").isNull(), F.lit(None))
        .when(
            F.col("morning_delay_rate_airport_day") > F.col("avg_morning_delay_rate_airport"),
            F.lit("high-delay morning")
        )
        .otherwise(F.lit("low-delay morning"))
    )

    df = df.drop("morning_delay_rate_airport_day","avg_morning_delay_rate_airport",
                                                    "dep_hour", "is_early_morning")
    
    ## Delay accumulation within the same day

    window_cum = Window.partitionBy("ORIGIN", "FL_DATE") \
                    .orderBy("CRS_DEP_TIME") \
                    .rowsBetween(Window.unboundedPreceding, -1)

    df = df.withColumn(
        "cum_delay_rate_today",
        F.avg(F.col("DEP_DEL15").cast('int')).over(window_cum)
    )

    print("Created cum_delay_rate_today")

    ## Yesterday's delay propagation

    # Daily delay rate by airport
    airport_daily_delay = (
        df.groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay"))
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Get yesterday's delay rate for each airport
    window_spec = Window.partitionBy("ORIGIN").orderBy("FL_DATE")

    airport_daily_delay = airport_daily_delay.withColumn(
        "yesterday_delay_rate",
        F.lag("airport_daily_delay_rate").over(window_spec)
    )

    # Classify yesterday vs that airport's historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        "day_before_delay_propagation",
        F.when(F.col("yesterday_delay_rate").isNull(), F.lit(None))
        .when(
            F.col("yesterday_delay_rate") > F.col("airport_historical_avg_delay"),
            F.lit("high-delay yesterday")
        )
        .otherwise(F.lit("low-delay yesterday"))
    )

    # Join feature back to original dataset
    df = df.join(
        #airport_daily_delay.select("ORIGIN", "FL_DATE", "day_before_delay_propagation"),
        airport_daily_delay.select("ORIGIN", "FL_DATE", "day_before_delay_propagation"),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created day_before_delay_propagation")

    ## 7-day delay propagation

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous 7 days by airport
    window_7d = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-7, -1)
    )

    # Compute average delay rate over previous 7 days
    airport_daily_delay = airport_daily_delay.withColumn(
        "previous_7d_delay_rate",
        F.avg("airport_daily_delay_rate").over(window_7d)
    )

    # Classify previous 7 days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        "7_day_delay_propagation",
        F.when(F.col("previous_7d_delay_rate").isNull(), F.lit(None))
        .when(
            F.col("previous_7d_delay_rate") > F.col("airport_historical_avg_delay"),
            F.lit("high-delay previous 7d")
        )
        .otherwise(F.lit("low-delay previous 7d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", "7_day_delay_propagation"),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created previous_7d_delay_rate")

    ## 15-day delay propagation

    days = 15

    prev_delay_col = f"previous_{days}d_delay_rate"
    propagation_col = f"{days}_day_delay_propagation"

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous X days by airport
    window_xd = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-days, -1)
    )

    # Compute average delay rate over previous X days
    airport_daily_delay = airport_daily_delay.withColumn(
        prev_delay_col,
        F.avg("airport_daily_delay_rate").over(window_xd)
    )

    # Classify previous X days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        propagation_col,
        F.when(F.col(prev_delay_col).isNull(), F.lit(None))
        .when(
            F.col(prev_delay_col) > F.col("airport_historical_avg_delay"),
            F.lit(f"high-delay previous {days}d")
        )
        .otherwise(F.lit(f"low-delay previous {days}d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", propagation_col),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created 15_day_delay_propagation")

    ## 30-day delay propagation

    days = 30

    prev_delay_col = f"previous_{days}d_delay_rate"
    propagation_col = f"{days}_day_delay_propagation"

    # Daily delay rate by airport
    airport_daily_delay = (
        df
        .groupBy("ORIGIN", "FL_DATE")
        .agg(
            F.avg(F.col("DEP_DEL15").cast("int")).alias("airport_daily_delay_rate")
        )
    )

    # Historical average delay rate for each airport
    airport_historical_avg = (
        airport_daily_delay
        .groupBy("ORIGIN")
        .agg(
            F.avg("airport_daily_delay_rate").alias("airport_historical_avg_delay")
        )
    )

    # Join airport historical average
    airport_daily_delay = airport_daily_delay.join(
        airport_historical_avg,
        on="ORIGIN",
        how="left"
    )

    # Window for previous X days by airport
    window_xd = (
        Window.partitionBy("ORIGIN")
        .orderBy("FL_DATE")
        .rowsBetween(-days, -1)
    )

    # Compute average delay rate over previous X days
    airport_daily_delay = airport_daily_delay.withColumn(
        prev_delay_col,
        F.avg("airport_daily_delay_rate").over(window_xd)
    )

    # Classify previous X days vs airport historical average
    airport_daily_delay = airport_daily_delay.withColumn(
        propagation_col,
        F.when(F.col(prev_delay_col).isNull(), F.lit(None))
        .when(
            F.col(prev_delay_col) > F.col("airport_historical_avg_delay"),
            F.lit(f"high-delay previous {days}d")
        )
        .otherwise(F.lit(f"low-delay previous {days}d"))
    )

    # Join feature back to original dataset
    df = df.join(
        airport_daily_delay.select("ORIGIN", "FL_DATE", propagation_col),
        on=["ORIGIN", "FL_DATE"],
        how="left"
    )

    print("Created 30_day_delay_propagation")

    ## Holiday indicator variable

    # US holidays
    years = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
    us_holidays = holidays.US(years=years)

    # Convert to list of strings (same format as FL_DATE)
    holiday_dates = [str(date) for date in us_holidays.keys()]

    df = df.withColumn(
        "is_us_holiday",
        F.when(F.col("FL_DATE").isin(holiday_dates), 1).otherwise(0)
    )

    print("Created is_us_holiday")

    ## Holiday proximity

    holidays_df = spark.createDataFrame(
        [(d,) for d in holiday_dates],
        ["holiday_date"]
    )

    #df = df.withColumn("FL_DATE", F.to_date("FL_DATE"))

    # Compute all the distances between each date to each holiday
    df_cross = df.crossJoin(F.broadcast(holidays_df))
    # Compute absolute differences in days
    df_cross = df_cross.withColumn(
        "days_diff",
        F.abs(F.datediff(F.col("FL_DATE"), F.col("holiday_date")))
    )
    # Get nearest holiday
    holiday_proximity = (
        df_cross.groupBy("FL_DATE")
                .agg(F.min("days_diff").alias("days_to_nearest_holiday"))
    )

    df = df.join(
        holiday_proximity.select("FL_DATE", "days_to_nearest_holiday"),
        on=["FL_DATE"],
        how="left"
    )

    print("Created days_to_nearest_holiday")

    ## Holiday proximity buckets

    df = df.withColumn(
        "holiday_proximity_bucket",
        F.when(F.col("days_to_nearest_holiday") == 0, "holiday")
        .when(F.col("days_to_nearest_holiday") <= 1, "±1 day")
        .when(F.col("days_to_nearest_holiday") <= 3, "±3 days")
        .when(F.col("days_to_nearest_holiday") <= 7, "±1 week")
        .otherwise("normal")
    )

    print("Created holiday_proximity_bucket")


    ## Fill nulls before encoding categorical engineered features

    categorical_cols = [
        "ORIGIN",
        "DEST",
        "ORIGIN_STATE_ABR",
        "DEST_STATE_ABR",
        "morning_delay_propagation",
        "day_before_delay_propagation",
        "7_day_delay_propagation",
        "15_day_delay_propagation",
        "30_day_delay_propagation",
        "holiday_proximity_bucket"
    ]

    numeric_fill_map = {
        "scheduled_flights_before_current": 0,
        "cum_delay_rate_today": 0.0,
        "days_to_nearest_holiday": 999
    }

    categorical_fill_map = {c: "missing" for c in categorical_cols}

    df = df.fillna(numeric_fill_map)
    df = df.fillna(categorical_fill_map)

    print("Filled up null values")

    ## Encode categorical engineered features
    ## Output:
    ## - *_idx  : indexed numeric category
    ## - *_ohe  : one-hot encoded vector for modeling

    indexed_cols = [f"{c}_idx" for c in categorical_cols]
    ohe_cols = [f"{c}_ohe" for c in categorical_cols]

    for c in categorical_cols:
        indexer = StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
        df = indexer.fit(df).transform(df)

    encoder = OneHotEncoder(
        inputCols=indexed_cols,
        outputCols=ohe_cols,
        handleInvalid="keep"
    )

    df = encoder.fit(df).transform(df)

    print("One-hot encoded features")
    
    df = df.drop(*indexed_cols)

    return df


# Weather

In [0]:
# weather_cols_string = [
#  "HourlyPresentWeatherType", #-> we can create binary columns like thunderstorm,rain,fog,snow from this column.
# "HourlySkyConditions", #-> BKN (Broken) and OVC (Overcast) are cloud cover categories
# ]
weather_cols_categorical = []

weather_cols_numeric = [
    "HourlyVisibility",
    "HourlyWindSpeed",
    "HourlyWindGustSpeed",
    "HourlyWindDirection",
    "HourlyPrecipitation",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyRelativeHumidity",
    "HourlyPressureChange",
    "HourlySeaLevelPressure",
    "HourlyPressureTendency",
]

selected_weather_cols = weather_cols_categorical + weather_cols_numeric

In [0]:
df_otpw_engr_weather = df_otpw_engr.select(weather_cols_categorical + weather_cols_numeric + ["MONTH", "FL_DATE", "DEP_DEL15"])
display(df_otpw_engr_weather)

In [0]:
df_otpw_engr_weather = cleanWeatherColumns(df_otpw_engr_weather, weather_cols_numeric = weather_cols_numeric, weather_cols_categorical = weather_cols_categorical)
df_otpw_engr_weather = impute_weather_columns(df_otpw_engr_weather)

## Model Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_weather.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_weather.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_weather.select(*(numeric_cols+["FL_DATE", "MONTH", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_w = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_w = scaler.fit(df_vec_w)
df_scaled_weather = scaler_model_w.transform(df_vec_w) #.select("scaled_features")

# Assemble features

# df_scaled_fin = df_scaled.drop("features") #I want to overwrite the features as it is already scaled in scaled_features
# feature_cols = ["YEAR", "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "QUARTER", "CRS_DEP_TIME", "CRS_ARR_TIME", "scaled_features"]
# assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
# df_model = assembler.transform(df_scaled_fin).withColumn("label", col("DEP_DEL15").cast("double"))

df_scaled_weather = df_scaled_weather.select("FL_DATE", "MONTH", "scaled_features", "label")
df_scaled_weather = df_scaled_weather.withColumnRenamed("scaled_features", "features")

##Checkpoint Data

In [0]:
display(df_scaled_weather, limit=10)

In [0]:

# Save df_etl as a parquet file
df_scaled_weather.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_weather_baseline.parquet")

# Flights

In [0]:
flights_cols_string = [
    "OP_UNIQUE_CARRIER",    #yes
    "ORIGIN",               #yes
    "DEST",                 #yes
    "ORIGIN_STATE_ABR",     #yes
    "DEST_STATE_ABR",       #yes
    "TAIL_NUM",             #yes
    "OP_CARRIER_FL_NUM",
]

flights_cols_categorical = [
    "QUARTER",             # Seasonal delay patterns
    "MONTH",               # Monthly delay variation
    "DAY_OF_MONTH",        # Day-of-month patterns
    "DAY_OF_WEEK",         # Day-of-week delay variation
    "YEAR",                # Year effects for multi-year datasets
    "CRS_DEP_TIME",        # Scheduled departure time — strong intra-day signal
    "CRS_ARR_TIME",        # Scheduled arrival time
]
# FL_DATE, 

flights_cols_numeric = [
    "DISTANCE",            # Route distance
]
selected_flights_cols = flights_cols_numeric + flights_cols_categorical + flights_cols_string

In [0]:
df_otpw_engr_flights = df_otpw_engr.select(selected_flights_cols + ["FL_DATE", "DEP_DEL15"])
display(df_otpw_engr_flights)

In [0]:
df_otpw_engr_flights = cleanFlightsColumns(df_otpw_engr_flights, flights_cols_numeric=flights_cols_numeric, flights_cols_categorical=flights_cols_categorical)

## Model Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_flights.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_flights.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_flights.select(*(numeric_cols+["FL_DATE", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_f = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model_f = scaler.fit(df_vec_f)
df_scaled_flights = scaler_model_f.transform(df_vec_f) #.select("scaled_features")


df_scaled_flights = df_scaled_flights.select("FL_DATE", "MONTH", "scaled_features", "label")
df_scaled_flights = df_scaled_flights.withColumnRenamed("scaled_features", "features")

##Checkpoint Data

In [0]:
display(df_scaled_flights, limit=10)

In [0]:

# Save df_etl as a parquet file
df_scaled_flights.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_flights_baseline.parquet")


# Baseline Weather + Flights + Engineered Flights

In [0]:
df_otpw_engr_baseline_wf = df_otpw_engr.select(selected_flights_cols + selected_weather_cols + ["FL_DATE", "DEP_DEL15"])
display(df_otpw_engr_baseline_wf)

In [0]:
df_otpw_engr_baseline_wf = cleanWeatherColumns(df_otpw_engr_baseline_wf, weather_cols_numeric = weather_cols_numeric, weather_cols_categorical = weather_cols_categorical)
df_otpw_engr_baseline_wf = impute_weather_columns(df_otpw_engr_baseline_wf)

df_otpw_engr_baseline_wf = cleanFlightsColumns(df_otpw_engr_baseline_wf, flights_cols_numeric=flights_cols_numeric, flights_cols_categorical=flights_cols_categorical)

## Model Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_baseline_wf.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_baseline_wf.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr = df_otpw_engr_baseline_wf.select(*(numeric_cols+["FL_DATE", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec = assembler.transform(df_corr).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_vec)
df_scaled_baseline = scaler_model.transform(df_vec) #.select("scaled_features")


df_scaled_baseline = df_scaled_baseline.select("FL_DATE", "MONTH", "scaled_features", "label")
df_scaled_baseline = df_scaled_baseline.withColumnRenamed("scaled_features", "features")

##Checkpoint Data

In [0]:
display(df_scaled_baseline, limit=10)

In [0]:

# Save df_etl as a parquet file
df_scaled_baseline.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_fw_baseline.parquet")

# Combined Baseline Weather + Flights

In [0]:
selected_flights_cols = flights_cols_numeric + flights_cols_categorical + flights_cols_string
df_otpw_engr_wf_f = df_otpw_engr.select(selected_flights_cols + selected_weather_cols + ["FL_DATE", "DEP_DEL15"])
display(df_otpw_engr_wf_f)

In [0]:
df_otpw_engr_wf_f = cleanWeatherColumns(df_otpw_engr_wf_f, weather_cols_numeric = weather_cols_numeric, weather_cols_categorical = weather_cols_categorical)
df_otpw_engr_wf_f = impute_weather_columns(df_otpw_engr_wf_f)

df_otpw_engr_wf_f = cleanFlightsColumns(df_otpw_engr_wf_f, flights_cols_numeric=flights_cols_numeric, flights_cols_categorical=flights_cols_categorical)

# df_otpw_engr_wf_f = featureEngineeredFlights(df_otpw_engr_wf_f)

## Model Readiness

In [0]:
numeric_cols = [c for c in df_otpw_engr_wf_f.columns if c != "DEP_DEL15" and c != "FL_DATE" and dict(df_otpw_engr_wf_f.dtypes)[c] != 'string']
numeric_cols

In [0]:
# Priyanka dummy checkpoint for Phase III
df_otpw_engr_wf_f.write.mode('overwrite').parquet(f"{folder_path}/testing/df_otpw_engr_baselinewf_prepackage.parquet")

In [0]:
# Normalization of the numeric features of the df_engr_mini_sample dataset using StandardScaler
df_corr_fe = df_otpw_engr_wf_f.select(*(numeric_cols+["FL_DATE", "DEP_DEL15"])).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
df_vec_fe = assembler.transform(df_corr_fe).withColumn("label", col("DEP_DEL15").cast("double"))

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_vec_fe)
df_scaled_wf_fe = scaler_model.transform(df_vec_fe) #.select("scaled_features")


df_scaled_wf_fe = df_scaled_wf_fe.select("FL_DATE", "MONTH", "scaled_features", "label")
df_scaled_wf_fe = df_scaled_wf_fe.withColumnRenamed("scaled_features", "features")

##Checkpoint Data

In [0]:
display(df_scaled_wf_fe, limit=10)

In [0]:

# Save df_etl as a parquet file
df_scaled_wf_fe.write.mode('overwrite').parquet(f"{folder_path}/delay_df_phase_2_fw_flights_engr.parquet")

#Sanity Check Models - Dummy Logistic Regression Test

In [0]:
weather_model, weather_metrics = run_logistic_regression(df_scaled_weather, features="features", label_col="label", seed=42)

In [0]:
flight_model, flight_metrics = run_logistic_regression(df_scaled_flights, features="features", label_col="label", seed=42)

In [0]:
base_model, base_metrics = run_logistic_regression(df_scaled_baseline, features="features", label_col="label", seed=42)

In [0]:
w_flights_engr_model, w_flights_engr_metrics = run_logistic_regression(df_scaled_wf_fe, features="features", label_col="label", seed=42)